<a href="https://colab.research.google.com/github/namii07/Namisha-Codeboosters-Internship-2026/blob/main/Phase_02_GenAI/Day_09_AIAgents_TextToSQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. AI Agents vs Chatbots

2. Tool Calling + Multi-Step Reasoning

3. Text to SQL Architecture

4. Building the Complete Pipeline

5. Mini Project

# AI AGENT -> Large Language Model(LLM)

  -Use tools

  -Take multiple steps

  -Decide which action to take

  -Act on the real world

### **Key Components of an AI Agent:**

| Component | Role | Example |
| --- | --- | --- |
| **LLM** | Brain / Decision Maker | Groq's LLaMA model |
| **Tools** | Actions the agent can take | SQL executor, web search |
| **Memory** | Past conversation context | Chat history |
| **Reasoning** | Multi-step planning | Chain of thought |

---

### **Chatbot vs AI Agent: Side-by-Side**

| Feature | Chatbot | AI Agent |
| --- | --- | --- |
| **Uses tools** | No | Yes |
| **Multi-step reasoning** | No | Yes |
| **Can query databases** | No | Yes |

---




# TOOL CALLING:
  
  It means giving the AI the ability to run specific Python fucntions.

  -> get_schema()

  -> generate_sql(question)

  -> execute_sql(query)


# MULTI-STEP REASONING : The ReAct Pattern

 **ReAct = Reasoning + Acting**

 The agent loops through four steps:

 1. THINK

 2. PLAN

 3. ACT

 4. RESPOND

# TEXT TO SQL - TECHNICAL ARCHITECTURE

Stage 1: Schema Injection

Stage 2: SQL generation

Stage 3: Execute and Respond

# PIPELINE:

1. Load CSV into SQLite

2. Get Db Schema

3. Generate SQL with Groq

4. Execute SQL on SQLite

5. Natural Language Answer

In [ ]:
!pip install groq  -q

print("Installed succesfully")

# Overriding the MODEL variable to resolve the NotFoundError
# The model 'llama-3.1-8b.instant' caused a NotFoundError.
# Changing to a known working model.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 1.3 MB/s eta 0:00:00
Installed succesfully


In [ ]:
import sqlite3

import pandas as pd

import os

from groq import Groq

import re

print("All libararies imported succcesfully")

All libararies imported succcesfully


In [ ]:
import os
os.environ["GROQ_API_KEY"] ="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"

client = Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL = "llama-3.1-8b-instant"

print("Groq client initialized succesfully")
print(f"Using model : {MODEL}")

Groq client initialized succesfully
Using model : llama-3.1-8b-instant


In [ ]:
# In a real session, students upload the provided CSV file
import io

csv_data = """student_id,name,age,gender,subject,marks,attendance,grade
1,Aarav Sharma,20,Male,Mathematics,88,92,A
2,Priya Patel,21,Female,Science,76,85,B
3,Rohan Mehta,20,Male,Programming,95,98,A+
4,Sneha Iyer,22,Female,Mathematics,62,78,C
5,Arjun Nair,21,Male,Programming,91,94,A+
6,Divya Krishnan,20,Female,Science,83,88,A
7,Karan Singh,22,Male,Mathematics,74,81,B
8,Ananya Gupta,21,Female,Programming,89,96,A
9,Vikram Reddy,20,Male,Science,70,79,B
10,Pooja Sharma,22,Female,Mathematics,55,72,D
11,Aditya Kumar,21,Male,Programming,97,99,A+
12,Meera Nambiar,20,Female,Science,81,87,A
13,Rahul Desai,22,Male,Mathematics,68,80,C
14,Kavitha Rajan,21,Female,Programming,86,93,A
15,Nikhil Verma,20,Male,Science,77,84,B
16,Swathi Pillai,22,Female,Mathematics,90,95,A+
17,Manish Joshi,21,Male,Programming,73,82,B
18,Lavanya Menon,20,Female,Science,66,76,C
19,Suresh Babu,22,Male,Mathematics,82,89,A
20,Anjali Singh,21,Female,Programming,94,97,A+
21,Deepak Nair,20,Male,Science,79,86,B
22,Rekha Sharma,22,Female,Mathematics,58,73,D
23,Sanjay Patel,21,Male,Programming,88,91,A
24,Usha Iyer,20,Female,Science,84,90,A
25,Vijay Kumar,22,Male,Mathematics,71,83,B
26,Nandita Rao,21,Female,Programming,92,96,A+
27,Ashok Reddy,20,Male,Science,65,77,C
28,Sunita Gupta,22,Female,Mathematics,87,93,A
29,Ravi Krishnan,21,Male,Programming,78,88,B
30,Bhavna Mehta,20,Female,Science,93,98,A+"""

df = pd.read_csv(io.StringIO(csv_data))

print(f"Dataset loaded: {len(df)} rows , {len(df.columns)} columns")
print("\nFirst 5 rows")

df.head()

Dataset loaded: 30 rows , 8 columns

First 5 rows


,student_id,name,age,gender,subject,marks,attendance,grade
0,1,Aarav Sharma,20,Male,Mathematics,88,92,A
1,2,Priya Patel,21,Female,Science,76,85,B
2,3,Rohan Mehta,20,Male,Programming,95,98,A+
3,4,Sneha Iyer,22,Female,Mathematics,62,78,C
4,5,Arjun Nair,21,Male,Programming,91,94,A+


In [ ]:
conn = sqlite3.connect("college.db")

df.to_sql("students" , conn, if_exists="replace", index=False)

print("Database created: college.db")
print("Table 'students' created with 30 student records")

test_df = pd.read_sql_query("SELECT COUNT(*) as total_rows FROM students", conn)
print(f"\n Verification: {test_df['total_rows'][0]} rows in database")



Database created: college.db
Table 'students' created with 30 student records

 Verification: 30 rows in database


In [ ]:
# Function to get the database schema (table structure)

def get_schema(conn, table_name="students"):
    """
    This function reads the structure of a database table.
    It returns information about each column: name and data type.

    Parameters:
        conn: The SQLite connection object (our link to the database)
        table_name: The name of the table to inspect (default: 'students')

    Returns:
        A formatted string describing the table structure
    """

    # Query SQLite's internal table info
    cursor = conn.cursor()
    # conn.cursor(): Creates a cursor object
    # A cursor is like a pointer that moves through the database
    # We use it to execute SQL commands

    cursor.execute(f"PRAGMA table_info({table_name})")
    # cursor.execute(): Runs a SQL command
    # PRAGMA table_info(): A special SQLite command that returns column information
    # It returns: column number, name, data type, nullable, default value, is primary key
    # f"...": An f-string — allows inserting variables inside the string using {}

    columns = cursor.fetchall()
    # cursor.fetchall(): Fetches ALL result rows from the last executed query
    # Returns a list of tuples, one tuple per column
    # Example tuple: (0, 'student_id', 'INTEGER', 0, None, 0)

    # Build a human-readable schema description
    schema_lines = [f"Table: {table_name}"]
    schema_lines.append("Columns:")

    for col in columns:
        # col[1]: column name (second element of the tuple)
        # col[2]: data type (third element)
        schema_lines.append(f"  - {col[1]} ({col[2]})")

    # Add sample values to help the AI understand the data
    cursor.execute(f"SELECT * FROM {table_name} LIMIT 3")
    # SELECT *: Select all columns
    # FROM {table_name}: From our students table
    # LIMIT 3: Return only the first 3 rows (so the schema text is not too long)

    sample_rows = cursor.fetchall()
    schema_lines.append("\nSample rows (first 3):")

    for row in sample_rows:
        schema_lines.append(f"  {row}")

    return "\n".join(schema_lines)
    # "\n".join(): Joins all lines with a newline character between them
    # This creates a single, multi-line string


# Test the schema function
schema = get_schema(conn)
print(schema)

Table: students
Columns:
  - student_id (INTEGER)
  - name (TEXT)
  - age (INTEGER)
  - gender (TEXT)
  - subject (TEXT)
  - marks (INTEGER)
  - attendance (INTEGER)
  - grade (TEXT)

Sample rows (first 3):
  (1, 'Aarav Sharma', 20, 'Male', 'Mathematics', 88, 92, 'A')
  (2, 'Priya Patel', 21, 'Female', 'Science', 76, 85, 'B')
  (3, 'Rohan Mehta', 20, 'Male', 'Programming', 95, 98, 'A+')


In [ ]:
def generate_sql(user_question, schema_text, client, model):

  system_prompt=f"""You are an expert SQL assistant. You are connected to a SQLite database with the following structure:

  {schema_text}

  Rules you must follow:
  1. Generate ONLY a valid SQLite SQL query.
  2. Do not include any explanation or  text -  only the SQL query.
  3. Do not use markdown code blocks. Return the raw SQL only.
  4. The table name is : students
  5. Only use column names that exist in the schema above.
  6. Use single quotes for string values in WHERE clauses( example: WHERE subject = 'Programming')
  7. If the user asks for top N , use ORDER BY marks DESC LIMIT N.

  """
  response=client.chat.completions.create(
      model=model,

      messages=[
          {"role":"system", "content":system_prompt},
          {"role":"user", "content":user_question}
      ],
      temperature=0.0
  )

  sql_query=response.choices[0].message.content.strip()

  return sql_query

question="Show me all female students"
print(f"Question : {question}")
print("\nGenerating SQL...")

sql=generate_sql(question, schema, client, MODEL)
print(f"\nGenerated SQL : \n{sql}")


Question : Show me all female students

Generating SQL...

Generated SQL : 
SELECT * FROM students WHERE gender = 'Female'


In [ ]:
def execute_sql(sql_query, conn):
  clean_sql = sql_query.strip()

  clean_sql = re.sub(r'```sql\s*', '' , clean_sql)

  clean_sql = clean_sql.strip()


  try:

    result_df = pd.read_sql_query(clean_sql,conn)

    return result_df , None

  except Exception as e:
    return None , str(e)


print(f"Executing SQL: {sql}")
result, error = execute_sql(sql, conn)

if error:
  print(f"Error: {error}")
else:
  print(f"\nQuery returned {len(result)} rows")
  print(result)

Executing SQL: SELECT * FROM students WHERE gender = 'Female'

Query returned 15 rows
    student_id            name  age  gender      subject  marks  attendance  \
0            2     Priya Patel   21  Female      Science     76          85   
1            4      Sneha Iyer   22  Female  Mathematics     62          78   
2            6  Divya Krishnan   20  Female      Science     83          88   
3            8    Ananya Gupta   21  Female  Programming     89          96   
4           10    Pooja Sharma   22  Female  Mathematics     55          72   
5           12   Meera Nambiar   20  Female      Science     81          87   
6           14   Kavitha Rajan   21  Female  Programming     86          93   
7           16   Swathi Pillai   22  Female  Mathematics     90          95   
8           18   Lavanya Menon   20  Female      Science     66          76   
9           20    Anjali Singh   21  Female  Programming     94          97   
10          22    Rekha Sharma   22  Female  

In [ ]:
# The complete Text-to-SQL Agent function

def text_to_sql_agent(user_question, conn, client, model, verbose=True):
    """
    The main AI Agent function.
    Takes a user question in plain English and returns database results.

    This is the complete workflow:
    1. Get database schema
    2. Generate SQL using Groq LLM
    3. Execute SQL on the database
    4. Format and return results

    Parameters:
        user_question: The question in plain English
        conn: SQLite database connection
        client: Groq API client
        model: Model name string
        verbose: If True, prints each step (useful for learning and debugging)

    Returns:
        A tuple: (result_dataframe, generated_sql_string)
    """

    print("=" * 60)
    # Prints a line of 60 equal signs as a visual divider
    # This makes the output easier to read

    print(f"USER QUESTION: {user_question}")
    print("=" * 60)

    # ---- STEP 1: Get the database schema ----
    if verbose:
        print("\n[STEP 1] Reading database schema...")

    schema_text = get_schema(conn)
    # Calls our earlier get_schema() function
    # This gives the AI the "map" of our database

    if verbose:
        print("Schema loaded successfully")

    # ---- STEP 2: Generate SQL ----
    if verbose:
        print("\n[STEP 2] Generating SQL query with Groq LLM...")

    generated_sql = generate_sql(user_question, schema_text, client, model)
    # Calls our generate_sql() function
    # Sends the question + schema to the AI
    # Gets back a SQL query string

    if verbose:
        print(f"Generated SQL:\n  {generated_sql}")

    # ---- STEP 3: Execute SQL on the database ----
    if verbose:
        print("\n[STEP 3] Executing SQL on the database...")

    result_df, error = execute_sql(generated_sql, conn)
    # Calls our execute_sql() function
    # Runs the generated SQL on the actual SQLite database
    # Returns result DataFrame and any error message

    if error:
        print(f"SQL Execution Error: {error}")
        return None, generated_sql
        # If there's an error, print it and return None for the result

    # ---- STEP 4: Display the results ----
    if verbose:
        print(f"\n[STEP 4] Query returned {len(result_df)} row(s)")
        print("\nRESULTS:")
        print("-" * 40)
        print(result_df.to_string(index=False))
        # .to_string(index=False): Converts DataFrame to formatted string without row numbers

    print("=" * 60)

    return result_df, generated_sql


# Test our complete agent
result, sql_used = text_to_sql_agent(
    "Show top 5 students in Programming",
    conn, client, MODEL
)

USER QUESTION: Show top 5 students in Programming

[STEP 1] Reading database schema...
Schema loaded successfully

[STEP 2] Generating SQL query with Groq LLM...
Generated SQL:
  SELECT name FROM students WHERE subject = 'Programming' ORDER BY marks DESC LIMIT 5

[STEP 3] Executing SQL on the database...

[STEP 4] Query returned 5 row(s)

RESULTS:
----------------------------------------
        name
Aditya Kumar
 Rohan Mehta
Anjali Singh
 Nandita Rao
  Arjun Nair


In [ ]:
# Test 1: Filtering by subject
result1, _ = text_to_sql_agent(
    "Show me all students who study Mathematics",
    conn, client, MODEL
)

USER QUESTION: Show me all students who study Mathematics

[STEP 1] Reading database schema...
Schema loaded successfully

[STEP 2] Generating SQL query with Groq LLM...
Generated SQL:
  SELECT * FROM students WHERE subject = 'Mathematics'

[STEP 3] Executing SQL on the database...

[STEP 4] Query returned 10 row(s)

RESULTS:
----------------------------------------
 student_id          name  age gender     subject  marks  attendance grade
          1  Aarav Sharma   20   Male Mathematics     88          92     A
          4    Sneha Iyer   22 Female Mathematics     62          78     C
          7   Karan Singh   22   Male Mathematics     74          81     B
         10  Pooja Sharma   22 Female Mathematics     55          72     D
         13   Rahul Desai   22   Male Mathematics     68          80     C
         16 Swathi Pillai   22 Female Mathematics     90          95    A+
         19   Suresh Babu   22   Male Mathematics     82          89     A
         22  Rekha Sharma   22 

In [ ]:
# Test 2: Aggregation (average)
result2, _ = text_to_sql_agent(
    "What is the average marks for each subject?",
    conn, client, MODEL
)

USER QUESTION: What is the average marks for each subject?

[STEP 1] Reading database schema...
Schema loaded successfully

[STEP 2] Generating SQL query with Groq LLM...
Generated SQL:
  SELECT subject, AVG(marks) FROM students GROUP BY subject

[STEP 3] Executing SQL on the database...

[STEP 4] Query returned 3 row(s)

RESULTS:
----------------------------------------
    subject  AVG(marks)
Mathematics        73.5
Programming        88.3
    Science        77.4


In [ ]:
# Test 3: Conditional filtering
result3, _ = text_to_sql_agent(
    "Show students who scored more than 90 marks",
    conn, client, MODEL
)

USER QUESTION: Show students who scored more than 90 marks

[STEP 1] Reading database schema...
Schema loaded successfully

[STEP 2] Generating SQL query with Groq LLM...
Generated SQL:
  SELECT * FROM students WHERE marks > 90

[STEP 3] Executing SQL on the database...

[STEP 4] Query returned 6 row(s)

RESULTS:
----------------------------------------
 student_id         name  age gender     subject  marks  attendance grade
          3  Rohan Mehta   20   Male Programming     95          98    A+
          5   Arjun Nair   21   Male Programming     91          94    A+
         11 Aditya Kumar   21   Male Programming     97          99    A+
         20 Anjali Singh   21 Female Programming     94          97    A+
         26  Nandita Rao   21 Female Programming     92          96    A+
         30 Bhavna Mehta   20 Female     Science     93          98    A+


In [ ]:
# Test 4: Counting
result4, _ = text_to_sql_agent(
    "How many male and female students are there?",
    conn, client, MODEL
)

USER QUESTION: How many male and female students are there?

[STEP 1] Reading database schema...
Schema loaded successfully

[STEP 2] Generating SQL query with Groq LLM...
Generated SQL:
  SELECT COUNT(CASE WHEN gender = 'Male' THEN 1 END) AS male_count, 
       COUNT(CASE WHEN gender = 'Female' THEN 1 END) AS female_count 
FROM students

[STEP 3] Executing SQL on the database...

[STEP 4] Query returned 1 row(s)

RESULTS:
----------------------------------------
 male_count  female_count
         15            15


In [ ]:
# Test 5: Complex query — multiple conditions
result5, _ = text_to_sql_agent(
    "Show female students who scored above 85 in Science or Programming, ordered by marks",
    conn, client, MODEL
)

USER QUESTION: Show female students who scored above 85 in Science or Programming, ordered by marks

[STEP 1] Reading database schema...
Schema loaded successfully

[STEP 2] Generating SQL query with Groq LLM...
Generated SQL:
  SELECT * FROM students WHERE gender = 'Female' AND subject IN ('Science', 'Programming') AND marks > 85 ORDER BY marks DESC

[STEP 3] Executing SQL on the database...

[STEP 4] Query returned 5 row(s)

RESULTS:
----------------------------------------
 student_id          name  age gender     subject  marks  attendance grade
         20  Anjali Singh   21 Female Programming     94          97    A+
         30  Bhavna Mehta   20 Female     Science     93          98    A+
         26   Nandita Rao   21 Female Programming     92          96    A+
          8  Ananya Gupta   21 Female Programming     89          96     A
         14 Kavitha Rajan   21 Female Programming     86          93     A


In [ ]:
# Function to generate a natural language response from the query results

def generate_answer(user_question, query_results_df, client, model):
    """
    Takes the original user question and the database results,
    then uses the LLM to write a friendly natural language answer.

    Parameters:
        user_question: The original question from the user
        query_results_df: The pandas DataFrame containing query results
        client: Groq API client
        model: Model name

    Returns:
        A string with the natural language answer
    """

    # Convert the DataFrame to a simple text format for the LLM
    if query_results_df is None or len(query_results_df) == 0:
        return "No results were found for your query."
    # Handle empty results: if no rows returned, give a clear message

    results_text = query_results_df.to_string(index=False)
    # .to_string(index=False): Converts the DataFrame table to a readable string
    # index=False: Hides the row numbers (0, 1, 2...)
    # This creates a text table that the LLM can read and understand

    # Prompt for natural language answer generation
    prompt = f"""The user asked: '{user_question}'

The database returned these results:
{results_text}

Please write a clear, friendly, 2-3 sentence answer to the user's question based on these results.
Be specific. Mention actual names and numbers from the data.
Do not add information not present in the results."""

    # Call the LLM with the results
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
        # temperature=0.3: Slightly more expressive than 0.0
        # For natural language answers, some variation is acceptable
        # But not too high — we still want accurate, factual answers
    )

    return response.choices[0].message.content.strip()


# Build the enhanced agent with natural language response

def smart_text_to_sql_agent(user_question, conn, client, model):
    """
    Enhanced agent that returns both a data table AND a natural language answer.
    """
    print("=" * 60)
    print(f"Question: {user_question}")
    print("=" * 60)

    # Step 1: Get schema
    schema_text = get_schema(conn)

    # Step 2: Generate SQL
    print("Generating SQL...")
    generated_sql = generate_sql(user_question, schema_text, client, model)
    print(f"SQL: {generated_sql}")

    # Step 3: Execute SQL
    result_df, error = execute_sql(generated_sql, conn)

    if error:
        print(f"Error executing SQL: {error}")
        return

    # Step 4: Show data table
    print(f"\nData ({len(result_df)} rows returned):")
    display(result_df)  # display() renders the DataFrame as a nice HTML table in Colab

    # Step 5: Generate natural language answer
    print("\nGenerating natural language answer...")
    answer = generate_answer(user_question, result_df, client, model)

    print("\nAnswer:")
    print(answer)
    print("=" * 60)


# Test the enhanced agent
smart_text_to_sql_agent(
    "Who are the top 5 students in Programming?",
    conn, client, MODEL
)

Question: Who are the top 5 students in Programming?
Generating SQL...
SQL: SELECT name FROM students WHERE subject = 'Programming' ORDER BY marks DESC LIMIT 5

Data (5 rows returned):


,name
0,Aditya Kumar
1,Rohan Mehta
2,Anjali Singh
3,Nandita Rao
4,Arjun Nair



Generating natural language answer...

Answer:
Based on the results, the top 5 students in Programming are Aditya Kumar, Rohan Mehta, Anjali Singh, Nandita Rao, and Arjun Nair. These students have secured the top spots in the programming course.


In [ ]:
# More tests with the enhanced agent

smart_text_to_sql_agent(
    "Which subject has the highest average attendance?",
    conn, client, MODEL
)

Question: Which subject has the highest average attendance?
Generating SQL...
SQL: SELECT subject FROM students GROUP BY subject ORDER BY AVG(attendance) DESC LIMIT 1

Data (1 rows returned):


,subject
0,Programming



Generating natural language answer...

Answer:
Based on the results, it appears that the subject with the highest average attendance is actually not specified in the results, as there is only one subject listed. However, since it's the only subject, we can say that 'Programming' has the highest average attendance with 1 subject.
